# Aula 09 — Laboratório
## Prever não é responder “e se?”

**Análise Econômica e Machine Learning** · Prof. Dr. Flavius Sodré · Economia, 8º período

Neste laboratório:

1. previsão ou efeito causal: seis perguntas e o 401(k);
2. a regressão com controles e a forma como a renda entra;
3. o DML à mão, num programa simulado com efeito conhecido;
4. o DML no 401(k), com três modelos;
5. o que o DML não resolve, no mundo da Aula 03.

No fim, a **nota técnica**: o 401(k) aumenta o patrimônio financeiro? Quanto, com que
intervalo e sob que hipótese?

**Como completar:** onde houver `# COMPLETE`, escreva o código. Onde houver `___`,
substitua. Rode as células na ordem.

## Bloco 0 — ambiente

In [ ]:
# Bloco 0 — rode esta celula e confira a saida.
import io, os, sys, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 120)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (10, 4.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# As mesmas cores dos slides. Azul = DML e quem fez o curso, laranja =
# regressao com controles lineares e quem nao fez, verde = uma sobra so.
# Bordo = o efeito verdadeiro.
AZUL, LARANJA, VERDE, BORDO, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#6F1D2B", "#8A8D96"

try:
    import sklearn
    from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
    from sklearn.linear_model import LassoCV
    from sklearn.model_selection import KFold
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import PolynomialFeatures, StandardScaler
except ImportError:
    raise ImportError("Este laboratorio precisa do scikit-learn. "
                      "Instale com:  pip install scikit-learn")

SEMENTE = 2026
print("Python      :", sys.version.split()[0], "|", platform.system())
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("\nAmbiente pronto.")

## Bloco 1 — previsão ou causa?

Para cada pergunta, escreva na tabela: **previsão** ou **efeito causal**, e uma frase sobre
o que mudaria no modelo ou nos dados.

| # | pergunta | tipo | o que muda |
|---|---|---|---|
| 1 | Quais clientes vão atrasar a fatura no mês que vem? | | |
| 2 | Se o banco aumentar o limite, o atraso cai? | | |
| 3 | Qual será o IPCA de dezembro? | | |
| 4 | Uma bolsa para quem fica na escola reduz a evasão no ensino médio? | | |
| 5 | Quais municípios de Pernambuco terão mais casos de dengue no verão? | | |
| 6 | Abrir uma agência bancária num município aumenta o emprego formal? | | |

In [ ]:
# Bloco 1 — Plano B. Rode e siga em frente; nao precisa ler.
#
# Resumo da base real (frequencias de idade, estudo e tamanho da familia;
# renda pela idade e pelo estudo; as demais caracteristicas e a elegibilidade
# pela renda; o patrimonio por uma regressao linear). Serve so para gerar
# familias inventadas com a mesma estrutura.

COLUNAS_401K = ['nifa', 'net_tfa', 'tw', 'age', 'inc', 'fsize', 'educ', 'db', 'marr', 'twoearn', 'e401', 'p401', 'pira', 'hown']

ESP_401K = {'age': {25: 0.0235,
         26: 0.0271,
         27: 0.0302,
         28: 0.0314,
         29: 0.0319,
         30: 0.0348,
         31: 0.0378,
         32: 0.0344,
         33: 0.0338,
         34: 0.0348,
         35: 0.0341,
         36: 0.0358,
         37: 0.033,
         38: 0.0359,
         39: 0.0307,
         40: 0.0337,
         41: 0.0308,
         42: 0.0345,
         43: 0.0331,
         44: 0.0301,
         45: 0.0255,
         46: 0.0281,
         47: 0.0225,
         48: 0.024,
         49: 0.0215,
         50: 0.0187,
         51: 0.0174,
         52: 0.0205,
         53: 0.0187,
         54: 0.0186,
         55: 0.0167,
         56: 0.0156,
         57: 0.0133,
         58: 0.0124,
         59: 0.0149,
         60: 0.0145,
         61: 0.0144,
         62: 0.0112,
         63: 0.0103,
         64: 0.01},
 'educ': {1: 0.0045,
          2: 0.001,
          3: 0.0027,
          4: 0.0034,
          5: 0.0034,
          6: 0.0125,
          7: 0.005,
          8: 0.016,
          9: 0.0179,
          10: 0.0309,
          11: 0.0309,
          12: 0.3768,
          13: 0.0856,
          14: 0.1137,
          15: 0.0452,
          16: 0.1329,
          17: 0.0325,
          18: 0.085},
 'fsize': {1: 0.2269,
           2: 0.2347,
           3: 0.1955,
           4: 0.2084,
           5: 0.0881,
           6: 0.0289,
           7: 0.0106,
           8: 0.0041,
           9: 0.0009,
           10: 0.0009,
           11: 0.0003,
           12: 0.0004,
           13: 0.0002},
 'log_renda': [7.489392, 0.09284, 0.068997, -0.000695],
 'dp_log_renda': 0.6548,
 'marr': [0.0467, 1.4539],
 'db': [-1.2115, 0.599],
 'pira': [-1.8491, 1.5074],
 'hown': [0.2606, 1.18],
 'e401': [-0.9682, 1.1656],
 'twoearn_dado_casado': 0.6291,
 'p401_dado_elegivel': 0.7045,
 'patrimonio': [-19383.7552, -0.5774, 612.1214, 219.8488, -1292.2897, 5392.0038, -14553.4256,
                -3230.6675, 30636.1176, 4638.4365, 9474.2032, 11.941],
 'dp_patrimonio': 54223.0}


def base_401k_sintetica(esp, n=9915, semente=0):
    """Famílias INVENTADAS com as mesmas colunas e tipos da base real: as
    frequências de idade, estudo e família, a renda pela idade e pelo estudo,
    as demais características e a elegibilidade pela renda, e o patrimônio por
    uma regressão linear com ruído de cauda pesada."""
    rng = np.random.default_rng(semente)

    def sorteia(freq):
        k = np.array([int(x) for x in freq])
        p = np.array(list(freq.values()), dtype=float)
        return rng.choice(k, n, p=p / p.sum())

    age, educ, fsize = sorteia(esp["age"]), sorteia(esp["educ"]), sorteia(esp["fsize"])
    b = esp["log_renda"]
    inc = np.round(np.exp(b[0] + b[1] * educ + b[2] * age + b[3] * age ** 2
                          + rng.normal(0, esp["dp_log_renda"], n)))
    inc = np.minimum(inc, 250000)
    lr = np.log(np.maximum(inc, 1000)) - 10

    def sim_nao(w):
        return (rng.random(n) < 1 / (1 + np.exp(-(w[0] + w[1] * lr)))).astype(int)

    marr, db, pira, hown, e401 = (sim_nao(esp[c]) for c in ("marr", "db", "pira", "hown", "e401"))
    twoearn = marr * (rng.random(n) < esp["twoearn_dado_casado"]).astype(int)
    p401 = e401 * (rng.random(n) < esp["p401_dado_elegivel"]).astype(int)
    X = np.column_stack([np.ones(n), inc, age, educ, fsize, marr, twoearn, db, pira, hown, e401,
                         (inc / 1000.0) ** 2])
    ruido = rng.standard_t(3, n) * esp["dp_patrimonio"] / np.sqrt(3)
    net_tfa = np.round(X @ np.array(esp["patrimonio"]) + ruido)
    nifa = np.maximum(np.round(net_tfa + rng.normal(0, 3000, n)), 0)
    tw = np.round(net_tfa + hown * rng.gamma(2.0, 30000, n))
    return pd.DataFrame({"nifa": nifa, "net_tfa": net_tfa, "tw": tw, "age": age, "inc": inc,
                         "fsize": fsize, "educ": educ, "db": db, "marr": marr,
                         "twoearn": twoearn, "e401": e401, "p401": p401, "pira": pira,
                         "hown": hown}).astype(int)


print("Plano B pronto (so e usado se a base real nao carregar).")

In [ ]:
# Bloco 1 — o 401(k): 9.915 familias americanas, SIPP de 1991 (Poterba,
# Venti e Wise, 1995; Chernozhukov e Hansen, 2004; Chernozhukov e outros, 2018).
#
# Tres caminhos, nesta ordem:
#   1. o arquivo 401k.csv, na pasta Dados do repositorio;
#   2. o arquivo original (sipp1991.dta) no repositorio de replicacao do DML;
#   3. o pacote `doubleml`, se estiver instalado.
# Se os tres falharem, segue o PLANO B (celula acima), com dados inventados.

ARQUIVO = "../Dados/401k.csv"
URL = "https://github.com/VC2015/DMLonGitHub/raw/master/sipp1991.dta"
CONTROLES = ['age', 'inc', 'educ', 'fsize', 'marr', 'twoearn', 'db', 'pira', 'hown']


def padronizar(d):
    """Mesmas colunas, na mesma ordem, em numeros inteiros."""
    d = d[COLUNAS_401K].round().astype(int)
    if len(d) < 1000:
        raise ValueError("a tabela nao tem a estrutura esperada")
    return d


def carregar_401k(arquivo=ARQUIVO, url=URL):
    if os.path.exists(arquivo):
        return padronizar(pd.read_csv(arquivo)), "arquivo local " + arquivo
    try:
        return padronizar(pd.read_stata(url)), "repositorio do DML (download)"
    except Exception as erro:
        print("download falhou:", type(erro).__name__, str(erro)[:80])
    try:
        from doubleml.datasets import fetch_401K
        return padronizar(fetch_401K(return_type="DataFrame")), "pacote doubleml"
    except Exception:
        pass
    print("=" * 70)
    print("PLANO B: os dados abaixo sao INVENTADOS. Mesmas colunas e tipos da")
    print("base real; familias e relacoes sao sinteticas. Os numeros NAO valem")
    print("como resultado sobre o 401(k).")
    print("=" * 70)
    return base_401k_sintetica(ESP_401K), "PLANO B (sintetico)"


familias, FONTE = carregar_401k()
print(f"fonte: {FONTE}")
print(f"{len(familias)} familias; {100 * familias['e401'].mean():.1f}% elegiveis ao 401(k)")
familias.head()

**Dicionário das colunas usadas**

| coluna | o que é |
|---|---|
| `net_tfa` | patrimônio financeiro líquido da família (US$): o **resultado** |
| `e401` | 1 se o empregador oferece o 401(k): o **tratamento** |
| `age`, `inc`, `educ`, `fsize` | idade, renda anual (US$), anos de estudo, tamanho da família |
| `marr`, `twoearn` | casado; dois com renda na família |
| `db`, `pira`, `hown` | tem pensão de benefício definido; tem conta IRA; casa própria |

O 401(k) é um plano de aposentadoria com desconto de imposto. Só existe se o **empregador**
o oferece: a família não escolhe ser elegível, mas escolhe onde trabalhar.

In [ ]:
# Bloco 1 — elegiveis e nao elegiveis. COMPLETE.
#
# A media de cada coluna nos dois grupos, lado a lado. Depois, a diferenca
# de medias do patrimonio.

# COMPLETE: media de net_tfa e dos CONTROLES por grupo de e401 (uma coluna por grupo)
comparacao = ___
comparacao.columns = ["nao elegiveis", "elegiveis"]
print(comparacao.round(2))

diferenca = comparacao.loc["net_tfa", "elegiveis"] - comparacao.loc["net_tfa", "nao elegiveis"]
print(f"\ndiferenca de medias do patrimonio: US$ {diferenca:,.0f}")

### Responda

> Em que os elegíveis diferem dos não elegíveis, além do 401(k)? Quanto da diferença de
> patrimônio você atribuiria ao plano, só com esta tabela?

## Bloco 2 — controles em linha reta

A regressão do patrimônio contra a elegibilidade e as nove características compara famílias
iguais nas características — **se** elas entram no patrimônio em linha reta.

In [ ]:
# Bloco 2 — a regressao com erro-padrao robusto. Leia, nao precisa alterar.

def ols(y, X, nomes):
    """Mínimos quadrados com constante e erro-padrão robusto (HC1).
    Devolve uma tabela com coeficiente e erro-padrão de cada coluna."""
    X = np.column_stack([np.ones(len(y)), np.asarray(X, dtype=float)])
    y = np.asarray(y, dtype=float)
    XtX_inv = np.linalg.pinv(X.T @ X)
    beta = XtX_inv @ X.T @ y
    u = y - X @ beta
    meio = (X * u[:, None]).T @ (X * u[:, None])
    v = XtX_inv @ meio @ XtX_inv * len(y) / (len(y) - X.shape[1])
    return pd.DataFrame({"coef": beta, "ep": np.sqrt(np.diag(v))},
                        index=["constante"] + list(nomes))


print("ols() pronta.")

In [ ]:
# Bloco 2 — a regressao. COMPLETE.

y = familias["net_tfa"]
# COMPLETE: as colunas da regressao — e401 primeiro, depois os CONTROLES
X = ___
reg = ols(y, X, ["e401"] + CONTROLES)
print(reg.round(1))

In [ ]:
# Bloco 2 — renda e idade em faixas. COMPLETE.

def faixas_401k(d=None):
    """As nove características com renda e idade em faixas: décimos de renda e
    faixas de cinco anos de idade, cada faixa uma coluna 0/1."""
    Z = d[["educ", "fsize", "marr", "twoearn", "db", "pira", "hown"]].astype(float).copy()
    renda = pd.qcut(d["inc"], 10, labels=False)
    idade = (d["age"] - 25) // 5
    for k in range(1, 10):
        Z[f"renda_{k}"] = (renda == k).astype(float)
    for k in range(1, int(idade.max()) + 1):
        Z[f"idade_{k}"] = (idade == k).astype(float)
    return Z


Z = faixas_401k(familias)
# COMPLETE: a mesma regressao, agora com as colunas de Z no lugar dos CONTROLES
reg_faixas = ___
print(f"em linha reta: {reg.loc['e401', 'coef']:,.0f} (ep {reg.loc['e401', 'ep']:,.0f})")
print(f"em faixas    : {reg_faixas.loc['e401', 'coef']:,.0f} (ep {reg_faixas.loc['e401', 'ep']:,.0f})")

### Responda

> O efeito da elegibilidade muda quando renda e idade entram em faixas? O que isso diz sobre
> a regressão em linha reta?

### ✔ CHECKPOINT I

A prefeitura tem duas perguntas: quais bairros vão ter fila por vaga em creche no ano que
vem, e se abrir vagas aumenta o emprego das mães.

Em duas frases: qual pede previsão e qual pede efeito causal? Para a segunda, por que
comparar bairros com e sem creche não basta?

## Bloco 3 — DML à mão

Num mundo simulado, o efeito verdadeiro é conhecido: R$ 300 por mês. Quatro estimativas,
da mais ingênua ao DML.

O DML usa **duas sobras**. A *sobra da renda* é a renda observada menos a que o modelo
prevê a partir das características; a *sobra do curso* é o mesmo para a participação (1 ou
0, menos a chance prevista). O efeito é a inclinação de uma sobra contra a outra. Cada
previsão vem de um modelo ajustado **nas outras dobras**.

In [ ]:
# Bloco 3 — o programa de qualificacao simulado. Leia, nao precisa alterar.
#
# Inscritos no sistema publico de emprego. O curso da prioridade a quem
# ganhava menos de R$ 2.200, a jovens e a quem esta ha mais tempo sem emprego;
# a renda depois nao fica abaixo do salario minimo. O efeito VERDADEIRO do
# curso e R$ 300 por mes para todos: e o numero a acertar.

EFEITO_PROGRAMA = 300.0
SALARIO_MINIMO = 1518.0
N_PROGRAMA = 4000
SEMENTE_PROGRAMA = 2026
COLUNAS_PROGRAMA = ['idade', 'escolaridade', 'metropole', 'mulher', 'renda_anterior', 'meses_desempregado']


def efeito_constante(dados):
    """O efeito do programa sobre a renda, pessoa a pessoa: R$ 300 para todos."""
    return np.full(len(dados), EFEITO_PROGRAMA)


def programa(n=N_PROGRAMA, semente=SEMENTE_PROGRAMA, efeito=efeito_constante):
    """Inscritos no sistema público de emprego, com e sem o curso de
    qualificação. A prioridade de vaga vai para quem ganhava menos de R$ 2.200
    (e menos ainda acima de R$ 3.500), para quem tem menos de 30 anos e para
    quem está há mais tempo sem emprego. A renda depois tem um piso: ninguém
    volta ao mercado formal ganhando menos que o salário mínimo.

    Devolve a tabela observada e, à parte, o que só o simulador conhece: a
    probabilidade de participar e o efeito de cada pessoa."""
    rng = np.random.default_rng(semente)
    idade = rng.integers(18, 61, n).astype(float)
    escolaridade = np.clip(np.round(rng.normal(10, 3.2, n)), 2, 17)
    metropole = (rng.random(n) < 0.55).astype(float)
    mulher = (rng.random(n) < 0.5).astype(float)
    a = idade - 18
    log_renda = (6.95 + 0.075 * escolaridade + 0.035 * a - 0.0006 * a ** 2
                 + 0.15 * metropole - 0.12 * mulher + rng.normal(0, 0.32, n))
    renda_anterior = np.round(np.exp(log_renda))
    meses = np.clip(np.round(rng.exponential(4 + 0.15 * a)), 0, 36)

    z = (0.6 - 2.2 * (renda_anterior > 2200) - 0.8 * (renda_anterior > 3500)
         + 0.9 * (idade < 30) + 0.06 * np.minimum(meses, 12) + 0.2 * metropole)
    p = np.clip(1 / (1 + np.exp(-z)), 0.03, 0.97)
    participou = (rng.random(n) < p).astype(float)

    dados = pd.DataFrame({"idade": idade, "escolaridade": escolaridade,
                          "metropole": metropole, "mulher": mulher,
                          "renda_anterior": renda_anterior, "meses_desempregado": meses})
    tau = efeito(dados)
    sem_programa = (np.maximum(SALARIO_MINIMO, 0.92 * renda_anterior)
                    + 0.00004 * np.maximum(renda_anterior - 2000, 0) ** 2
                    - 22 * np.minimum(meses, 24) + 160 * metropole - 120 * mulher
                    + 8 * a - 0.2 * a ** 2)
    ruido = rng.normal(0, 450, n)
    dados["programa"] = participou
    dados["renda_depois"] = np.round(sem_programa + tau * participou + ruido)
    oculto = pd.DataFrame({"prob_participar": p, "efeito": tau,
                           "renda_sem_programa": sem_programa + ruido})
    return dados, oculto


inscritos, oculto = programa()
print(f"{len(inscritos)} inscritos; {100 * inscritos['programa'].mean():.1f}% fizeram o curso")
inscritos.head()

In [ ]:
# Bloco 3 — o modelo das sobras e a sobra unica. Leia, nao precisa alterar.

def modelo_boosting():
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1, max_depth=3,
                                         min_samples_leaf=20, random_state=0)


def uma_sobra(X, y, D, fabrica, dobras=5, semente=SEMENTE):
    """O atalho que parece DML e não é: prevê a renda pelas características e
    compara participantes e não participantes no que sobrou. Falta a segunda
    sobra — a do tratamento."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    D = np.asarray(D, dtype=float)
    sobra_y = np.empty(len(y))
    for treino, teste in KFold(dobras, shuffle=True, random_state=semente).split(X):
        sobra_y[teste] = y[teste] - fabrica().fit(X[treino], y[treino]).predict(X[teste])
    Dc = D - D.mean()
    return float(Dc @ sobra_y / (Dc @ Dc))


print("Prontos: modelo_boosting() e uma_sobra().")

In [ ]:
# Bloco 3 — a funcao dml(). COMPLETE as tres lacunas.

def dml(X, y, D, fabrica, dobras=5, semente=SEMENTE):
    """Double/debiased machine learning no modelo parcialmente linear.

    Duas sobras, com dobras cruzadas: a renda que o modelo não previu a partir
    das características e o tratamento que o modelo não previu a partir das
    mesmas características. Cada previsão vem de um modelo ajustado nas outras
    dobras, que não viu aquela linha. O efeito é a inclinação de uma sobra
    contra a outra; o erro-padrão é o robusto dessa regressão."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    D = np.asarray(D, dtype=float)
    sobra_y = np.empty(len(y))
    sobra_d = np.empty(len(y))
    for treino, teste in KFold(dobras, shuffle=True, random_state=semente).split(X):
        # COMPLETE: ajuste o modelo nas dobras de treino e preveja a dobra de teste
        sobra_y[teste] = ___       # renda: o observado menos o previsto
        sobra_d[teste] = ___       # tratamento: o observado menos o previsto
    # COMPLETE: a inclinacao de uma sobra contra a outra
    efeito = ___
    u = sobra_y - efeito * sobra_d
    ep = float(np.sqrt(np.sum((sobra_d * u) ** 2)) / (sobra_d @ sobra_d))
    return {"efeito": efeito, "ep": ep, "sobra_y": sobra_y, "sobra_d": sobra_d}


print("dml() pronta.")

In [ ]:
# Bloco 3 — quatro estimativas. COMPLETE.

X, y, D = inscritos[COLUNAS_PROGRAMA], inscritos["renda_depois"], inscritos["programa"]

dif = y[D == 1].mean() - y[D == 0].mean()
reg_p = ols(y, np.column_stack([D, X]), ["programa"] + COLUNAS_PROGRAMA)
# COMPLETE: a sobra unica e o DML, os dois com modelo_boosting
uma = ___
r = ___

print(f"diferenca de medias : {dif:8.0f}")
print(f"regressao com contr.: {reg_p.loc['programa', 'coef']:8.0f}")
print(f"uma sobra so        : {uma:8.0f}")
print(f"DML                 : {r['efeito']:8.0f}  (intervalo de 95%: "
      f"{r['efeito'] - 1.96 * r['ep']:.0f} a {r['efeito'] + 1.96 * r['ep']:.0f})")
print(f"VERDADEIRO          : {EFEITO_PROGRAMA:8.0f}")

In [ ]:
# Bloco 3 — vinte mundos. COMPLETE.
#
# Para cada semente de 0 a 19: gere o programa, calcule a sobra unica e o
# DML (efeito e erro-padrao). Quantos intervalos de 95% cobrem os R$ 300?

linhas = []
for semente in range(20):
    ins, _ = programa(semente=semente)
    X_, y_, D_ = ins[COLUNAS_PROGRAMA], ins["renda_depois"], ins["programa"]
    # COMPLETE
    r_ = ___
    linhas.append({"uma_sobra": uma_sobra(X_, y_, D_, modelo_boosting),
                   "dml": r_["efeito"], "ep": r_["ep"]})

vinte = pd.DataFrame(linhas)
vinte["cobre"] = (vinte["dml"] - 1.96 * vinte["ep"] <= 300) & (300 <= vinte["dml"] + 1.96 * vinte["ep"])
print(vinte[["uma_sobra", "dml"]].describe().loc[["mean", "std", "min", "max"]].round(0))
print("intervalos que cobrem 300:", vinte["cobre"].sum(), "de 20")

### Responda

> Quem acerta em média: a sobra única ou o DML? O intervalo do DML cobre os R$ 300 tantas
> vezes quanto prometia?

## Bloco 4 — DML no 401(k)

A mesma função `dml()`, agora no 401(k), com três modelos para as sobras: um Lasso com as
colunas, seus quadrados e produtos; uma floresta; e o boosting.

In [ ]:
# Bloco 4 — os tres modelos. Leia, nao precisa alterar.

def modelo_floresta():
    return RandomForestRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.5,
                                 n_jobs=-1, random_state=0)


def modelo_lasso():
    """Lasso com as colunas, seus quadrados e os produtos dois a dois."""
    return make_pipeline(PolynomialFeatures(2, include_bias=False), StandardScaler(),
                         LassoCV(alphas=np.logspace(-2, 4, 40), cv=5, max_iter=20000))


MODELOS_DML = {"lasso": modelo_lasso, "floresta": modelo_floresta, "boosting": modelo_boosting}
print("Prontos:", ", ".join(MODELOS_DML))

In [ ]:
# Bloco 4 — a tabela. COMPLETE. (A floresta leva uns 20 segundos.)

X4, y4, D4 = familias[CONTROLES], familias["net_tfa"], familias["e401"]
tabela = {"diferenca de medias": {"efeito": diferenca, "ep": np.nan},
          "regressao com controles": {"efeito": reg.loc["e401", "coef"], "ep": reg.loc["e401", "ep"]}}
for nome, fabrica in MODELOS_DML.items():
    # COMPLETE
    r4 = ___
    tabela["DML " + nome] = {"efeito": r4["efeito"], "ep": r4["ep"]}

tabela = pd.DataFrame(tabela).T
tabela["de"] = tabela["efeito"] - 1.96 * tabela["ep"]
tabela["ate"] = tabela["efeito"] + 1.96 * tabela["ep"]
print(tabela.round(0))

In [ ]:
# Bloco 4 — sem a renda. COMPLETE.
#
# O DML com o boosting, tirando "inc" dos controles.

sem_renda = [c for c in CONTROLES if c != "inc"]
# COMPLETE
r_sem = ___
print(f"com a renda: {tabela.loc['DML boosting', 'efeito']:,.0f}")
print(f"sem a renda: {r_sem['efeito']:,.0f} (ep {r_sem['ep']:,.0f})")

### Responda

> O que acontece com o efeito quando a renda sai dos controles? Por quê?

## Bloco 5 — o que o DML não resolve

No mundo da Aula 03, o efeito verdadeiro de um ano de estudo é conhecido. O DML com quatro
conjuntos de controles: qual acerta?

In [ ]:
# Bloco 5 — o mundo da Aula 03. Leia, nao precisa alterar.
#
# Uma pesquisa domiciliar ficticia. A HABILIDADE afeta o estudo e o salario e
# ninguem a observa; o cargo de chefia e CONSEQUENCIA do estudo. A coluna
# _habilidade existe porque o mundo e simulado.

EFEITO_DIRETO = 0.06
EFEITO_HABILIDADE = 0.26
EFEITO_EXPERIENCIA = 0.03
EFEITO_EXPERIENCIA2 = -0.0005
EFEITO_CARGO = 0.8
RUIDO = 0.33
CORR_ORIGEM_HAB = 0.92
SETORES = ('Serviços', 'Indústria', 'Comércio', 'Público')
EFEITO_SETOR = {'Serviços': 0.0, 'Indústria': 0.1, 'Comércio': -0.08, 'Público': 0.18}
REGIOES = ('Nordeste', 'Sudeste', 'Sul', 'Norte', 'Centro-Oeste')
EFEITO_REGIAO = {'Nordeste': 0.0, 'Sudeste': 0.2, 'Sul': 0.14, 'Norte': 0.02, 'Centro-Oeste': 0.1}


def gerar_base(n=4000, semente=42, anos_a_mais=0.0):
    """Uma pesquisa domiciliar fictícia de rendimentos.

    Estrutura de uma PNAD reduzida: uma linha por trabalhador, salário mensal,
    anos de estudo, experiência, origem familiar, setor, região e ocupação de
    chefia. A HABILIDADE existe no mundo e entra no salário, mas NÃO é
    observada — é ela que faz a regressão ingênua mentir.

    `anos_a_mais` faz a intervenção: dá a todo mundo esse tanto de estudo a
    mais, mantendo habilidade, sorteios e ruídos idênticos. É assim que o
    efeito causal verdadeiro é medido, e não lido de um parâmetro.
    """
    rng = np.random.default_rng(semente)

    # habilidade: existe, afeta estudo e salário, e ninguém mede
    hab = rng.normal(0, 1, n)

    # origem familiar (escolaridade dos pais, padronizada): observada, e
    # correlacionada com a habilidade — por isso serve de proxy imperfeita
    origem = (CORR_ORIGEM_HAB * hab
              + np.sqrt(1 - CORR_ORIGEM_HAB ** 2) * rng.normal(0, 1, n))

    # anos de estudo: quem tem mais habilidade e melhor origem estuda mais
    estudo = 9.0 + 1.00 * origem + 1.00 * hab + rng.normal(0, 1.8, n)
    estudo = np.clip(np.round(estudo), 4, 20) + anos_a_mais

    # experiência: anos de trabalho, independente do estudo neste mundo
    experiencia = np.clip(np.round(rng.normal(18, 9, n)), 0, 40)

    setor = rng.choice(SETORES, n, p=[0.45, 0.20, 0.22, 0.13])
    regiao = rng.choice(REGIOES, n, p=[0.30, 0.34, 0.14, 0.08, 0.14])

    # cargo de chefia: CONSEQUÊNCIA do estudo e da habilidade. Controlar por
    # ele fecha parte do caminho pelo qual o estudo aumenta o salário.
    indice = 0.85 * (estudo - 11) / 3 + 0.70 * hab + 0.03 * experiencia
    cargo = (indice + rng.normal(0, 1, n) > 1.10).astype(int)

    log_salario = (
        6.40
        + EFEITO_DIRETO * estudo
        + EFEITO_EXPERIENCIA * experiencia
        + EFEITO_EXPERIENCIA2 * experiencia ** 2
        + EFEITO_HABILIDADE * hab
        + EFEITO_CARGO * cargo
        + np.array([EFEITO_SETOR[s] for s in setor])
        + np.array([EFEITO_REGIAO[r] for r in regiao])
        + rng.normal(0, RUIDO, n)
    )

    return {
        "salario": np.round(np.exp(log_salario), 2),
        "log_salario": log_salario,
        "estudo": estudo,
        "experiencia": experiencia,
        "origem": np.round(origem, 3),
        "setor": setor,
        "regiao": regiao,
        "cargo": cargo,
        "_habilidade": hab,          # não entregar ao aluno
    }


def efeito_causal_verdadeiro(n=200000, semente=2024):
    """Mede o efeito de UM ano a mais de estudo intervindo no mundo.

    Gera a mesma população duas vezes — com os mesmos sorteios de habilidade,
    setor, região e ruído — e na segunda dá um ano de estudo a mais a todos.
    A diferença média no log do salário é o efeito causal total, incluindo a
    parte que passa por virar chefe. Ele NÃO está escrito em nenhuma linha
    do modelo.
    """
    a = gerar_base(n=n, semente=semente, anos_a_mais=0.0)
    b = gerar_base(n=n, semente=semente, anos_a_mais=1.0)
    return float(np.mean(b["log_salario"] - a["log_salario"]))


def tabela_aula03(base=None):
    """O mundo da Aula 03 como tabela, com setor e região em colunas 0/1."""
    b = gerar_base() if base is None else base
    t = pd.DataFrame({k: b[k] for k in ("log_salario", "estudo", "experiencia", "origem",
                                         "cargo", "_habilidade")})
    for s in SETORES[1:]:
        t[f"setor_{s}"] = (b["setor"] == s).astype(float)
    for r in REGIOES[1:]:
        t[f"regiao_{r}"] = (b["regiao"] == r).astype(float)
    return t


def colunas_do_conjunto(t, conjunto):
    cols = []
    for c in conjunto:
        cols += [k for k in t.columns if k == c or k.startswith(c + "_")]
    return cols


CONJUNTOS = {'experiência, setor e região': ['experiencia', 'setor', 'regiao'],
 '+ origem familiar': ['experiencia', 'setor', 'regiao', 'origem'],
 '+ origem e cargo de chefia': ['experiencia', 'setor', 'regiao', 'origem', 'cargo'],
 'experiência, setor, região e habilidade': ['experiencia', 'setor', 'regiao', '_habilidade']}

trabalhadores = tabela_aula03()
VERDADEIRO = efeito_causal_verdadeiro()
print(f"{len(trabalhadores)} trabalhadores; efeito verdadeiro de 1 ano de estudo: "
      f"{100 * VERDADEIRO:.1f}%")

In [ ]:
# Bloco 5 — quatro conjuntos. COMPLETE.

resultado = {}
for nome, conjunto in CONJUNTOS.items():
    cols = colunas_do_conjunto(trabalhadores, conjunto)
    # COMPLETE: o DML do estudo no log do salario, com essas colunas e modelo_boosting
    r5 = ___
    resultado[nome] = {"efeito (%)": 100 * r5["efeito"], "ep (%)": 100 * r5["ep"]}

print(pd.DataFrame(resultado).T.round(1))
print(f"\nVERDADEIRO: {100 * VERDADEIRO:.1f}%")

### ✔ CHECKPOINT II

Com a floresta, o DML estima que ser elegível ao 401(k) aumenta o patrimônio financeiro em
cerca de US$ 8,7 mil.

Em duas frases: o que precisa ser verdade para esse número ser um efeito causal? Dê um
exemplo de algo não observado que poderia derrubá-lo.

## A nota técnica

Em até cinco frases: o 401(k) aumenta o patrimônio financeiro das famílias? Quanto, com que
intervalo, e sob que hipótese? Cite a estimativa que sustenta cada afirmação.

_(escreva aqui)_